![Platform+](images/banner.png)

# LAB 4: CREDITS & COST AWARENESS
_Hands-on Snowflake for data practitioners_

> **Persona track: Data Engineer** — same lab, your angle.

👉 **In this lesson, we'll understand, monitor, and estimate your Snowflake compute costs.**

📚 **By the end you'll be able to:**
- Identify what consumes Snowflake credits
- Read warehouse and query usage from ACCOUNT_USAGE
- Estimate cost before scaling a workload
- Right-size a warehouse and set auto-suspend

**Why this matters for engineers:** Your pipelines run on warehouses you configure. A poorly sized warehouse or a missing auto-suspend on a scheduled task can burn credits 24/7 — and you're the one who'll be asked to fix it.

**Time:** ~35 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('\"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · What Costs Money?
Snowflake bills primarily on two axes: **compute** (credits consumed by warehouses and serverless features, billed by the second) and **storage** (per TB/month, usually a much smaller line item). There are additional billing dimensions (cloud services, data transfer) covered in the [full pricing documentation](https://docs.snowflake.com/en/user-guide/credits) — for this lab, we focus on the two that make up 90%+ of most bills.

The table below shows what draws on the compute meter.

> 💡 **What's a credit?** A Snowflake **credit** is the unit of compute consumption — the currency you spend while warehouses run and serverless features work. Your account draws down credits from a prepaid contract or an on-demand balance, and their dollar value depends on your edition, cloud, and region. Everything in this lab is ultimately measured in credits — for reference, an X-Small warehouse burns **1 credit per hour**.

| Service | When it burns credits |
|---|---|
| Virtual warehouse | Every second it runs (60-second minimum each time it resumes) |
| Snowpipe (serverless) | Per GB ingested (fixed credits/GB) |
| Scheduled tasks (serverless) | Per execution |
| Cortex AI functions | Per token / API call |
| Storage | Per TB/month (on daily-average storage) — a separate, lower meter |

> 📌 **Your #1 cost lever:** a warehouse costs credits *only while it runs* — an idle, auto-suspended warehouse costs **zero**. Setting auto-suspend well is the simplest, highest-leverage control you have.

## 2 · Monitor Your Costs (Part 1)
Snowflake's [`SNOWFLAKE.ACCOUNT_USAGE`](https://docs.snowflake.com/en/sql-reference/account-usage) schema is a built-in telemetry database of everything that has happened in your account. Three telemetry areas cover most cost investigation work.

> 🔐 **Access note.** `ACCOUNT_USAGE` is restricted by default — a data engineer normally wouldn't have it. For this lab your role has been granted two read-only Snowflake database roles: **`USAGE_VIEWER`** (warehouse & credit usage) and **`GOVERNANCE_VIEWER`** (query history). Snowflake recommends these fine-grained database roles over the older, broad `IMPORTED PRIVILEGES` grant precisely so you don't hand out more access than you intend. In production, this access is typically an admin or governance-team privilege.

> ⏱️ **Expect a lag.** `ACCOUNT_USAGE` views are not real-time — they refresh on a delay that varies by view ([see current latencies](https://docs.snowflake.com/en/sql-reference/account-usage#account-usage-views)): `WAREHOUSE_METERING_HISTORY` up to ~3 hours, `QUERY_HISTORY` up to ~45 minutes, `QUERY_ATTRIBUTION_HISTORY` up to ~8 hours, and the Cortex usage views up to ~1 hour. An empty or incomplete result in a fresh training account is normal.

### 2a · Warehouse-level spend
[`WAREHOUSE_METERING_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/warehouse-metering-history) records credits consumed per warehouse, aggregated by hour. The query below rolls the last 7 days up per warehouse.

In [ ]:
SELECT warehouse_name,
       ROUND(SUM(credits_used), 3) AS credits_7d
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** Which warehouse used the most credits? An empty result just means the account is quiet — normal in a fresh training environment.

### 2b · Per-query cost

**Why this matters for engineers:** This is how you identify which pipeline step (MERGE, CTAS, transform) is your most expensive — the evidence for optimizing or splitting stages.
[`QUERY_ATTRIBUTION_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history) records a credit cost for each individual query. It does **not** include query text — for that you join to `QUERY_HISTORY`. Together they let you find the most expensive pipeline steps or ad-hoc queries.

In [ ]:
-- A query can appear as several rows in QUERY_ATTRIBUTION_HISTORY (e.g. if it
-- spans a warehouse resize), so we SUM credits per query_id first, then join
-- to QUERY_HISTORY for the text and execution details.
WITH qa AS (
    SELECT query_id,
           MAX(warehouse_name)              AS warehouse_name,
           SUM(credits_attributed_compute)  AS credits_attributed_compute
    FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
    WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
    GROUP BY query_id
)
SELECT LEFT(qh.query_text, 70)                  AS query_preview,
       qa.warehouse_name,
       ROUND(qa.credits_attributed_compute, 5)  AS credits,
       ROUND(qh.execution_time / 1000.0, 1)      AS secs,
       ROUND(qh.bytes_scanned / 1024 / 1024, 1)  AS mb_scanned,
       qh.query_type
FROM qa
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
  ON qa.query_id = qh.query_id
ORDER BY credits DESC
LIMIT 10;

> 🔎 **Review the output.** Credits at the query level are small numbers — a few thousandths is typical for an XSMALL warehouse. But multiply by frequency: a query costing 0.001 credits that runs 1,000 times/day = 1 credit/day = 30 credits/month. The value here is **relative comparison** — which pipeline steps cost more, and how often do they run?

### 2c · AI and Cortex service costs

**Why this matters for engineers:** If your pipeline calls Cortex AI functions (AI_COMPLETE, AI_CLASSIFY), this view separates that cost from warehouse compute — useful for estimating per-record AI processing budgets.
Snowflake provides dedicated ACCOUNT_USAGE views for each AI service. Here are the current-generation views to use (older views like `CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY` are deprecated):

| View | What it tracks |
|---|---|
| [`CORTEX_AI_FUNCTIONS_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history) | All Cortex AI SQL functions (AI_COMPLETE, AI_CLASSIFY, AI_SENTIMENT, etc.) — per query, function, and model |
| [`CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_code_snowsight_usage_history) | Cortex Code (CoCo) usage inside Snowsight |
| [`CORTEX_ANALYST_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history) | Cortex Analyst (natural language to SQL) |
| [`CORTEX_AGENT_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_agent_usage_history) | Cortex Agents API |

These usage views refresh with a latency of up to ~1 hour (see the access & latency note in Section 2). **We'll explore Cortex AI functions themselves in Lab 5** — here we're only looking at their *cost*. 👉 **Try the example below:**

In [ ]:
-- AI function credit consumption by function and model (last 7 days)
SELECT function_name,
       model_name,
       ROUND(SUM(credits), 4)         AS credits_7d,
       COUNT(DISTINCT query_id)        AS queries
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** An empty result means no Cortex AI functions ran in this period — normal in a fresh training account. In production, this query tells you exactly which functions and models are driving AI credit spend, down to the individual query if needed. Different models have vastly different per-token costs — see the [Cortex AI Functions cost reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost) for current rates by model and function.

**CoCo (Cortex Code) usage in Snowsight.** The query below sums CoCo usage **over the last 7 days**, one row per user. It reads the source columns `TOKEN_CREDITS` (credit cost), `TOKENS` (volume) and `USAGE_TIME`, and returns them under friendlier aliases you'll see in the result: **`CREDITS_7D`**, **`TOKENS_7D`** and **`REQUESTS`** (a count of CoCo calls).

What's a **token**? Language models don't read whole words — they break text into *tokens*, chunks of roughly ¾ of a word (about 4 characters of English). Both your prompt and the model's response are measured in tokens, and token volume is what drives AI credit cost. `TOKENS_7D` is the total tokens processed; `CREDITS_7D` converts that volume into credits.

In [ ]:
-- CoCo usage inside Snowsight (chat, SQL generation, code assist)
SELECT user_name,
       ROUND(SUM(token_credits), 4) AS credits_7d,
       SUM(tokens)                  AS tokens_7d,
       COUNT(*)                     AS requests
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY
WHERE usage_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** An empty result means no CoCo Snowsight sessions ran in this 7-day period. **`CREDITS_7D`** is the credit cost, **`TOKENS_7D`** the token volume, and **`REQUESTS`** the number of CoCo calls. For context: a typical CoCo conversation (a few prompts + responses) uses roughly 2,000–5,000 tokens — so if you see `TOKENS_7D` in the hundreds of thousands, that represents many interactions across the period.

### 2d · Snowsight Cost Management
Snowsight has a built-in **Cost Management** page that shows the same data without SQL. Navigate to **Admin → Cost Management** and use the controls:

- **Consumption tab, Usage Type = Compute**: warehouse credit trend by day — the same data as `WAREHOUSE_METERING_HISTORY`, visualised
- **Consumption tab → set the *View by* control (top-right of the chart) to *By Service*:** consumption splits into warehouse usage plus each serverless feature (e.g. `WAREHOUSE_METERING`, `PIPE`, and the serverless Cortex/AI features) as its own line — serverless spend visible without SQL
- **Account Overview tab**: summary cards for total spend, budget utilisation, anomaly alerts

> 📌 **What you cannot see in the UI:** per-query credit attribution is not surfaced in Snowsight. `QUERY_ATTRIBUTION_HISTORY` (Section 2b) is the only way to get cost at the individual query level.

## 3 · Estimate Before You Build (Part 2)
Monitoring tells you what *has* happened. Estimation helps you plan *before* you build. Snowflake recommends selecting a starting warehouse size based on **workload type and query complexity**, not on row count alone — then adjusting based on observed performance signals.

### How Snowflake recommends selecting warehouse size

From the [Warehouse considerations](https://docs.snowflake.com/en/user-guide/warehouses-considerations) docs: **start with the size that matches your workload type, then adjust based on observed performance.** Per-second billing means you can upsize, run a job, and suspend without committing permanently. And 'larger is not necessarily faster' for small, basic queries.

**Starting size by workload type:**

| Workload | Starting size | Key sizing factor |
|---|---|---|
| Development, testing, PoC | X-Small | Sufficient for exploration and Snowsight UI queries |
| Interactive analytics (small datasets) | X-Small – Small | Single-table queries, light filters |
| Regular reporting (multi-table JOINs, GBs scanned) | Small – Medium | Query complexity and bytes scanned |
| Heavy ELT, large transforms | Medium – Large | Data volume in bytes, not row count |
| Very large-scale production workloads | Large – X-Large | Experiment across sizes and compare runtime |

**Credit rates.** The table below shows the published **Gen1** per-hour rates (the in-docs baseline). **Gen2 standard warehouses are now the default for new accounts in most AWS/Azure regions** — they run on faster hardware at a **higher credit rate: roughly ~35% more on AWS/GCP and ~25% more on Azure** than the Gen1 figures below. Existing accounts may still be on Gen1 unless explicitly upgraded. The doubling-per-size-step pattern still holds within Gen2's own rates. Because Gen2 finishes many queries faster it consumes fewer seconds, so whether it costs more *in practice* depends on the workload — Snowflake's guidance is to **test, don't assume faster is cheaper**. For exact current numbers see the [Gen2 warehouses documentation](https://docs.snowflake.com/en/user-guide/warehouses-gen2), which links the Snowflake Service Consumption Table.

| Size | Credits/hr (Gen1 ref) | Credits/sec |
|---|---|---|
| X-Small | 1 | 0.0003 |
| Small | 2 | 0.0006 |
| Medium | 4 | 0.0011 |
| Large | 8 | 0.0022 |
| X-Large | 16 | 0.0044 |

### Adaptive Warehouses — sizing without the guesswork
**Adaptive Warehouses** are a newer warehouse type (Enterprise Edition, select AWS regions) that remove the sizing decision entirely. Instead of you picking XS/S/M and hand-tuning auto-suspend, Snowflake **allocates compute per query automatically** and scales resources up or down to match each workload as it runs.

- **Different billing model:** you're charged per query based on the compute actually consumed — not per-second for a fixed-size warehouse that may sit partly idle. (Per-query credit usage for Adaptive Warehouses shows up in the `QUERY_METERING_HISTORY` view.)
- **Good fit:** varied, unpredictable analytical workloads and mixed ELT + BI, where right-sizing by hand is a constant chore.
- **Not for:** HTAP (hybrid table) workloads or Snowpark-optimised (ML / memory-heavy) workloads.

> 📌 If Adaptive Warehouses are available in your region, evaluate them before manually sizing — they can remove the size-tuning cycle for mixed analytical workloads. The manual-sizing guidance in this lab still applies to standard (Gen1/Gen2) warehouses, which remain the default elsewhere.

**Step 1 — see how much data this workload touches.** Now that you know how to read cost telemetry and what signals to watch for, you're ready to estimate costs for a workload *before* you run it. The first input is data volume: how many rows does your workload scan? Run the query below to find out for the Roastery orders table.

> 📌 Table size in bytes and query complexity matter more than row count when choosing a warehouse size — but row count gives you a useful order-of-magnitude anchor.

In [ ]:
-- Data volume context for this workload
SELECT COUNT(*) AS row_count
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

> 🔎 **Review the output.** The Roastery dataset is small. For the estimation exercise, use the workload-type table above — pick the tier that matches how you'd use this data (exploration vs. reporting vs. heavy ELT), not a tier derived from this row count.

👉 **Step 2 — estimate monthly credits.** Pick the workload tier from the table above, set your expected active hours, and run the cell.

In [ ]:
# YOUR TURN: pick your starting workload tier and working hours, then run
# Step 1 — choose a tier from the workload-type table above
workload_tier        = 1   # <- 1=X-Small  2=Small  3=Medium  4=Large  5=X-Large

# Step 2 — estimate your working hours per day
hours_active_per_day = 8   # <- how many hours per day the warehouse is running

# -- official Gen1 credit rates (don't change below this line) ----------
WH_SIZES = {1: ('X-Small', 1), 2: ('Small', 2), 3: ('Medium', 4), 4: ('Large', 8), 5: ('X-Large', 16)}
wh_size, cph = WH_SIZES.get(workload_tier, ('X-Large', 16))
monthly_credits = cph * hours_active_per_day * 30

print(f'Starting WH size:      {wh_size}')
print(f'Credits/hr:            {cph}')
print(f'Est. monthly credits:  {monthly_credits}  ({hours_active_per_day}h/day x 30 days)')
print()
print('This is a starting estimate. Run your actual workload, check for spilling in')
print('QUERY_HISTORY.bytes_spilled_to_local_storage, and resize based on those signals.')

> 🔎 **Review the output.** Try changing the tier or hours: what does a Medium warehouse running 12h/day cost vs an X-Small running 8h/day?

👉 **Step 3 — convert to dollars.** Your estimate shows monthly credits. To see the dollar cost for your cloud provider and region:

[**Snowflake Pricing Calculator →**](https://www.snowflake.com/en/pricing-options/calculator)

Enter your cloud provider, region, and the monthly credits from the estimator above. This gives you a realistic cost range before you build.

> 📌 **Note your estimate.** Write down or screenshot your recommended WH size and monthly credits from above — this is your starting point for cost discussions and warehouse configuration decisions.

## 4 · Control Cost: Right-size + Auto-suspend

**Why this matters for engineers:** Scheduled pipelines have predictable run patterns — you can size the warehouse to the workload and set auto-suspend to match the gap between runs, minimizing idle burn.
Two warehouse settings that are always worth checking:

- **`WAREHOUSE_SIZE`** — bigger warehouses cost more per hour but finish jobs faster. Match the size to the workload, not the other way around.
- **`AUTO_SUSPEND`** — idle timeout in seconds before the warehouse suspends itself. `60` seconds is a sensible default for interactive workloads.

**Why not to set auto-suspend too aggressively:** a suspended warehouse drops its local disk cache. The first few queries after resume run cold — slower until the cache rebuilds. If your workload has regular short gaps (e.g. a dashboard refreshing every 2 minutes), aggressive auto-suspend triggers repeated suspend-resume cycles, each billed a minimum 60 seconds. Match the suspend window to your actual inter-query gap pattern.

**Auto-resume:** a warehouse with `AUTO_RESUME = TRUE` (the default) restarts automatically the moment any query is issued against it — typically 1–2 seconds. Users see no error; they simply wait slightly longer for the first query after an idle period.

Set both on your own warehouse now.

In [ ]:
ALTER WAREHOUSE {{my_wh}} SET AUTO_SUSPEND = 60;
ALTER WAREHOUSE {{my_wh}} SET WAREHOUSE_SIZE = XSMALL;
SHOW WAREHOUSES LIKE '{{my_wh}}';

> 🔎 **Review the output.** AUTO_SUSPEND should now read 60. Small settings like this, applied consistently across a fleet, add up.

### Spot under-sizing: check for spilling

**After you've run your workload,** confirm your sizing choice with evidence. The clearest sign a warehouse is **too small** is **spilling**: when a query runs out of memory it writes intermediate results to disk (local first, then remote). Learn more: [What spilling is and why it hurts](https://community.snowflake.com/s/article/Performance-impact-from-local-and-remote-disk-spilling) · [Finding and fixing spilling queries](https://docs.snowflake.com/en/user-guide/performance-query-warehouse-memory). Spilling is one tool in your right-sizing arsenal, and you can see it two ways:

- **In the UI:** open a query's **Query Profile** and read the **Bytes spilled to local/remote storage** statistics.
- **Programmatically:** query the `bytes_spilled_to_*` columns in `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` (the cell below).

> 💡 **First line of defense: read the query, not just the size.** Before upsizing, check the query itself — pull only the columns and rows you need, and confirm every join has a condition. A poorly-written query (missing filter, accidental cross join) is a common root cause of spilling; a bigger warehouse just pays to move the same waste faster.

How to read the result:

- `spill_local_mb > 0`: mildly undersized — upsize one step
- `spill_remote_mb > 0`: severely undersized — upsize 2+ steps
- **No rows**: no queries spilled — the warehouse is *big enough* for current workloads (on its own this doesn't prove it isn't over-sized; use the metering + query-cost views to check for waste)

In [ ]:
-- Check for any spilling in the last 7 days (the signal to upsize)
SELECT LEFT(query_text, 150)              AS query,
       warehouse_name,
       ROUND(bytes_spilled_to_local_storage / 1024 / 1024, 2)    AS spill_local_mb,
       ROUND(bytes_spilled_to_remote_storage / 1024 / 1024, 2)   AS spill_remote_mb,
       query_acceleration_bytes_scanned AS qas_bytes,
       ROUND(execution_time / 1000.0, 1) AS secs
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND (bytes_spilled_to_local_storage > 0
    OR bytes_spilled_to_remote_storage > 0)
ORDER BY bytes_spilled_to_local_storage DESC
LIMIT 10;

> 🔎 **Review the output.** No rows = nothing spilled, so the warehouse is big enough (not necessarily right-sized — an over-sized warehouse also shows no spill; cross-check with the metering/query-cost views). If you do see rows, read the two spill columns differently:
>
> | Column | Severity | What it means |
> |--------|----------|---------------|
> | `spill_local_mb` > 0 | ⚠️ Minor | Query overflowed memory to local SSD — slower but tolerable for occasional large queries |
> | `spill_remote_mb` > 0 | 🔴 Significant | Query overflowed to cloud storage — major performance hit, strong signal to upsize the warehouse |
>
> A non-zero **qas_bytes** means Query Acceleration Service helped that query — spill on a QAS-accelerated query is still a real undersizing signal, but note QAS was already compensating for it.
>
> **Rule of thumb:** Occasional local spill on your largest query is acceptable. Any remote spill, or frequent local spill, means the warehouse is undersized for that workload — step the size up and re-check.

### Try a CoCo skill: `/warehouse`

Skills were introduced in Lab 2. Here's the first skill use in this lab. `/warehouse` has structured knowledge of Gen2 architecture, credit billing mechanics, sizing trade-offs, and auto-suspend patterns.

👉 Open the **CoCo panel**, type `/` to open the skill picker, select **warehouse**.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/warehouse For a data engineer running 10 queries per day against the Roastery orders table (small dataset, XSMALL warehouse with AUTO_SUSPEND=60), is this the right configuration? What would you recommend for a mixed workload of ad-hoc queries and scheduled daily reports?
</pre>

> 📌 **What to expect:** The skill validates the configuration, explains Gen2 billing mechanics, and advises on the auto-suspend vs cache warmth trade-off.

## 5 · CoCo Cost Intelligence — Ask the AI
CoCo has a built-in **cost-intelligence** skill that runs ACCOUNT_USAGE queries for you and adds a plain-English summary — the same data as Sections 2a–2c, surfaced through AI.

**How to invoke the skill:**
1. Open the **CoCo panel** (chat icon, top-right in Snowsight)
2. Type `/` — a skill picker appears
3. Select **cost-intelligence**
4. Ask your questions

👉 **Try these three questions in order:**

**Q1 — Biggest spenders:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence What are my top 3 warehouses by credit spend in the last 30 days?
</pre>

> 📌 After CoCo answers, find the SQL it generated (you may need to unfurl its execution steps), click **Copy**, paste it into the cell below, amend the date range to show the last **90 days**, and add `LIMIT 10`. Run it. Is your warehouse in this list?

In [ ]:
-- YOUR TURN: paste CoCo's SQL on the blank line below, then:
-- 1. Change the date range to 90 days (e.g. DATEADD(day, -90, CURRENT_TIMESTAMP()))
-- 2. Add LIMIT 10 at the end to show the top 10 results


**Q2 — Drill into a warehouse:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence How many credits per day is {{my_wh}} consuming and what can I do to reduce it?
</pre>

**Q3 — AI and service costs:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence Show me credit spend by service type in the last 30 days, including any Cortex AI or CoCo usage.
</pre>

> 🔎 **Review CoCo's output.** Notice it generates SQL, executes it, and adds a summary. The skill is doing exactly what you did manually in Sections 2a–2c — this is how AI assistance accelerates the cost-monitoring workflow.

## 6 · Your Angle — Data Engineer
Attribute credit cost to specific pipeline steps, isolate cost by tag, and use `/workload-performance-analysis` to check for pipeline-level spilling.

**Find your most expensive recurring pipeline patterns:**
Pipelines run the *same query shape* on a schedule — so the DE cost question isn't "which single query was most expensive?" (that's the analyst view from Section 2) but **"which recurring *pattern*, across all its runs, cost the most — and how often does it run?"** `QUERY_ATTRIBUTION_HISTORY` carries the per-query credit cost plus a `QUERY_PARAMETERIZED_HASH` that is identical across every run of the same query shape. Group by that hash to roll cost up per pattern. (Credit cost lives in `QUERY_ATTRIBUTION_HISTORY`; the query text lives in `QUERY_HISTORY` — so we join the two, the same join you used in Section 2.)

In [ ]:
-- Recurring pipeline patterns: same query shape, ranked by TOTAL credits across all runs
SELECT ANY_VALUE(LEFT(qh.query_text, 80))            AS example_step,
       COUNT(*)                                       AS runs,
       ROUND(SUM(qah.credits_attributed_compute), 5)  AS total_credits,
       ROUND(AVG(qah.credits_attributed_compute), 6)  AS avg_credits_per_run
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY qah
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
  ON qah.query_id = qh.query_id
WHERE qah.start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY qah.query_parameterized_hash
ORDER BY total_credits DESC
LIMIT 15;

> 🔎 **Review the output.** The `runs` column is the pipeline insight: a cheap query fired hundreds of times can outrank a single heavy one on `total_credits` — that's the step to optimise first. Grouping by `query_parameterized_hash` collapses every run of the same query shape into one row, ignoring the literal values that change between runs.

**The DE multiplier — why a "cheap" step still hurts:**
A query that costs a fraction of a credit looks harmless — until you remember a pipeline runs it *on a schedule, forever*. The number that matters to a data engineer isn't the cost of one run, it's **`per-run cost × runs per day × 30 days`**. Take the `avg_credits_per_run` from your top pattern above, decide how often it runs, and project the monthly cost.

In [ ]:
# YOUR TURN: project a recurring pipeline step's monthly cost
avg_credits_per_run = 0.0058   # <- paste avg_credits_per_run from the pattern above
runs_per_day        = 288      # <- how often the pipeline runs it (288 = every 5 minutes)

# ---- projection (don't change below this line) ----
monthly = avg_credits_per_run * runs_per_day * 30
hourly  = avg_credits_per_run * 24 * 30
print(f'Per run:              {avg_credits_per_run} credits')
print(f'Runs per day:         {runs_per_day}')
print(f'Monthly credits:      {round(monthly, 2)}   ({runs_per_day}/day x 30 days)')
print('-' * 52)
print(f'Same step, hourly:    {round(hourly, 2)} credits/month  (24/day x 30)')
print()
print('The query did not change — only how often it runs. Cadence is a DE cost lever.')

> 🔎 **Review the output.** Look at the gap between the every-5-minute and hourly totals — *same query, very different bill*. As a DE you don't just run queries, you **schedule** them, so before setting a cadence ask: does this really need to run this often? Right-sizing *frequency* is as powerful as right-sizing the warehouse — you'll put this into practice with tasks and dynamic tables later in the course, and tie them together in the capstone.

**Tag a query, then find it in attribution history:**
Setting a `QUERY_TAG` before a pipeline step makes cost attribution clean. `QUERY_ATTRIBUTION_HISTORY` has a `QUERY_TAG` column directly — no join required to filter by tag.

👉 Run the tagged query below. This sets your session tag, runs the aggregation, then unsets it.

> ⏱️ **Heads-up on latency:** your freshly tagged query won't appear in `QUERY_ATTRIBUTION_HISTORY` for up to ~8 hours (ACCOUNT_USAGE lag), so don't expect to find it there during class. However, you *can* confirm your tagged query ran immediately using the real-time table function:
>
> ```sql
> SELECT query_id, query_tag, execution_status
> FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
> WHERE query_tag = '{{user}}_LAB4_DE_DEMO' ORDER BY start_time DESC LIMIT 5;
> ```
>
> This has no latency — it proves the tag worked. Credit attribution just needs time to propagate.
>
> 👉 **Try it:** Click **+ SQL** in the toolbar to add a SQL cell below, paste the query above, and run it.
>
> That's exactly why the next step uses a query the account **pre-tagged for you at setup time** — so you can see real attribution *now* instead of waiting.

In [ ]:
ALTER SESSION SET QUERY_TAG = '{{user}}_LAB4_DE_DEMO';
-- A tagged pipeline step — visible in QUERY_ATTRIBUTION_HISTORY
SELECT store_id, COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS revenue
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS
GROUP BY store_id
ORDER BY revenue DESC;
ALTER SESSION UNSET QUERY_TAG;

> 🔎 **Review the output.** Your tagged query ran successfully — but remember the latency note above: it takes up to ~8 hours to appear in `QUERY_ATTRIBUTION_HISTORY`, so don't expect to find it there during class. The next cell uses the account's **pre-tagged** query so you can see real attribution *now*. The workflow is the same: tag every pipeline step → filter by tag → get per-job cost.

**Isolate the cost of a pre-tagged query:**
The account setup script ran a query tagged `PLATPLUS_LAB4_DEMO` at provisioning time — run early so it has time to propagate into `QUERY_ATTRIBUTION_HISTORY` before class. The query below looks it up directly using the `QUERY_TAG` column.

> ⏱️ **Empty result?** If the setup script's tagged query hasn't propagated to `QUERY_ATTRIBUTION_HISTORY` yet (up to ~8 hours after provisioning), this query returns empty. That's expected on a freshly provisioned account — the pattern is what matters, not seeing this specific row.

In [ ]:
-- Credit cost of the setup-tagged query (tagged during account provisioning)
SELECT ROUND(SUM(credits_attributed_compute), 6) AS credits_attributed,
       COUNT(DISTINCT query_id)                   AS executions,
       MIN(start_time)                            AS first_seen
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
WHERE query_tag = 'PLATPLUS_LAB4_DEMO';

> 🔎 **Review the output.** `credits_attributed` is the credit cost of that tagged step. In production the tag is the pipeline name — this one-line filter is your per-job cost report.

**Group all queries by tag — cost per job or team:**

In [ ]:
SELECT COALESCE(query_tag, '(no tag)') AS tag,
       COUNT(*)                         AS queries,
       ROUND(SUM(credits_attributed_compute), 5) AS total_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY total_credits DESC
LIMIT 10;

> 🔎 **Review the output.** If you see a `(no tag)` row — that's unattributed cost (queries that ran without a QUERY_TAG set). Any other rows represent tagged, attributable work. In a fresh training account you may see *only* `(no tag)` since most lab queries aren't tagged — that's expected. In production, the goal is to shrink the untagged row toward zero by tagging every pipeline step and scheduled job.

**Cost-aware pipeline design patterns:**

| Pattern | Why it saves credits |
|---|---|
| Right-size per job type | XSMALL for light ELT; MEDIUM only for heavy transforms |
| Batch after business hours | Avoid peak concurrency; larger WHs run fewer hours |
| 60s auto-suspend between steps | Stops idle billing during gaps in a pipeline |
| `SET QUERY_TAG` before each step | Makes attribution easier; trace cost to specific jobs |
| Avoid `SELECT *` in pipelines | Pull only the columns needed to reduce data scanned |

Now put that tag data to work. Instead of asking CoCo *how* to tag, ask it to **act on the tags you already have** — find your most expensive tagged job and tell you where to cut it. That's chargeback and optimization in one step.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence Which of my tagged pipeline jobs is the most expensive over the last 30 days, and what specific changes would reduce its credit consumption?
</pre>

### Supporting: check for spilling with the performance skill.
Spilling to disk is a cost and performance issue simultaneously — it means the warehouse ran out of memory and wrote intermediate results to storage, slowing the query and burning extra credits.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/workload-performance-analysis analyze {{my_wh}} for any spilling in the last 7 days
</pre>

> 📌 **What to expect:** The skill checks `bytes_spilled_to_disk` and partition scan ratios. High spilling means the WH is undersized for that query's memory needs — a signal to right-size the warehouse or rewrite the query.

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q4")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q5")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q6")

### Key takeaways. ❄️
- Compute burns credits; a *suspended* warehouse costs nothing (a running-but-idle one still bills) — auto-suspend is your simplest lever
- WAREHOUSE_METERING_HISTORY and QUERY_ATTRIBUTION_HISTORY show where credits go
- Match warehouse size to data volume and workload type before scaling
- Estimate monthly credits before building; use the Snowflake Pricing Calculator to convert to dollars

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.